In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random

%matplotlib inline

In [ ]:
root = Path('..')
data_path = root / 'data' / 'names.txt'
words = open(data_path).read().splitlines()

print(f'Words count: {len(words)}')

In [ ]:
# build vocabulary of characters and mappings to/from ints

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
print(itos)

In [ ]:
# build dataset split into train / val / test

def build_dataset(words, block_size=3):
    X, Y = [], []

    for w in words:

        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, X.dtype, Y.shape, Y.dtype)
    return X, Y

X, Y = build_dataset(words)

s1 = int(0.8 * len(X))
s2 = int(0.9 * len(X))

X_train, Y_train = X[:s1], Y[:s1]
X_val, Y_val = X[s1:s2], Y[s1:s2]
X_test, Y_test = X[s2:], Y[s2:]

print(f'train: {len(X_train)} val: {len(X_val)} test: {len(X_test)}')

In [ ]:
n_embed = 10
n_hidden = 200
vocab_size = len(stoi)
block_size = 3
g = torch.Generator().manual_seed(42)
C = torch.randn((vocab_size, n_embed), generator=g) # 27 characters, increased embedding size to 10 dimensions
print(C.shape)

W1 = torch.randn((n_embed * block_size, n_hidden), generator=g) * (5/3) / ((n_embed * block_size) ** 0.5)
b1 = torch.randn(n_hidden, generator=g) * 0.025
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.05
b2 = torch.randn(vocab_size, generator=g) * 0

parameters = [C, W1, b1, W2, b2]

total_params = sum(p.nelement() for p in parameters) # total number of parameters
print(f'Total parameters: {total_params}')

for p in parameters: p.requires_grad = True # enable gradients

In [ ]:
EPOCHS = 100_000
BATCH_SIZE = 64

LR = 0.135

lri = []
lossi = []
stepi = [] # let's also track the training steps

In [ ]:
# Training loop
# use cross-entropy loss

for epoch in range(EPOCHS):

    # minibatch
    indices = torch.randint(0, len(X_train), (BATCH_SIZE,), generator=g)

    # forward pass
    emb = C[X_train[indices]]
    hpreact = emb.view(-1, 30) @ W1 + b1
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y_train[indices])
    if epoch % 5_000 == 0: print(f'epoch: {epoch} loss: {loss.item()}')

    # backward pass
    for p in parameters: p.grad = None
    loss.backward()

    # gradient descent step 
    lr = LR if epoch < 50_000 else LR / 10 # weight decay after 125k steps
    for p in parameters: p.data -= p.grad * lr

    # track loss
    lri.append(lr)
    lossi.append(loss.log10().item())
    stepi.append(epoch)

In [ ]:
# plt.plot(stepi, lossi)

In [ ]:
# plt.hist(h.view(-1).tolist(), 50) # histogram of hidden layer activations

In [ ]:
# plt.hist(hpreact.view(-1).tolist(), 50) # histogram of hidden layer pre-activations

In [ ]:
plt.figure(figsize=(20, 10))
plt.imshow(h.abs() > 0.99, cmap='gray', interpolation='nearest') # visualize the activations of the hidden layer

In [ ]:
@torch.no_grad()
def split_loss(split):
    x, y = {
        'train': (X_train, Y_train),
        'val': (X_val, Y_val),
        'test': (X_test, Y_test)
    }[split]
    emb = C[x]
    embcat = emb.view(emb.shape[0], -1)
    h = torch.tanh(embcat @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y)
    print(f'{split} loss: {loss.item()}')

split_loss('train')
split_loss('val')
split_loss('test')